## 1. SlidesGo Template

In [14]:
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import pandas as pd
import time

def scrape_slidescarnival_selenium(num_pages=3, delay=2):
    options = webdriver.ChromeOptions()
    options.binary_location = r"C:/Program Files/Google/Chrome/Application/chrome.exe"
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')

    service = Service(r"C:/Users/naeun/capstone/chromedriver-win64/chromedriver.exe")
    driver = webdriver.Chrome(service=service, options=options)
    all_data = []

    for page in range(1, num_pages + 1):
        url = f"https://www.slidescarnival.com/category/free-templates/powerpoint-templates?page={page}"
        print(f"Scraping page {page} … URL: {url}")
        driver.get(url)
        time.sleep(delay)

        # 각 템플릿 카드
        cards = driver.find_elements(By.CSS_SELECTOR, "div.image-wrapper._16_9-landscape")
        print(f"Found {len(cards)} cards on page {page}")

        for card in cards:
            try:
                template_url = card.get_attribute("data-url")  # 상세페이지 링크

                # 템플릿 상세페이지로 이동
                driver.get(template_url)
                time.sleep(1)

                # 제목
                title_img = driver.find_element(By.CSS_SELECTOR, "div.swiffy-slider img")
                title = title_img.get_attribute("title") or "No title"

                # 모든 슬라이드 이미지 URL 수집
                slide_imgs = driver.find_elements(By.CSS_SELECTOR, "ul.slider-container li img")
                img_urls = [img.get_attribute("src") for img in slide_imgs if img.get_attribute("src").endswith((".jpg", ".png"))]

                # Canva / PPT / Google Slides 링크
                canva_link = driver.find_element(By.CSS_SELECTOR, "a.sc-canva").get_attribute("href")
                ppt_link = driver.find_element(By.CSS_SELECTOR, "a.sc-powerpoint").get_attribute("href")
                google_link = driver.find_element(By.CSS_SELECTOR, "a.sc-googleslides").get_attribute("href")

                all_data.append({
                    "title": title,
                    "template_url": template_url,
                    "slide_imgs": img_urls,
                    "canva": canva_link,
                    "ppt": ppt_link,
                    "google": google_link
                })

                driver.back()
                time.sleep(1)

            except Exception as e:
                print("Skipped a template:", e)
                driver.back()
                continue

    driver.quit()
    return all_data

def save_to_csv(data_list, fname="slidescarnival_dataset.csv"):
    if not data_list:
        print("⚠️ No data to save.")
        return
    df = pd.DataFrame(data_list)
    df.to_csv(fname, index=False, encoding="utf-8-sig")
    print(f"✅ Saved {len(data_list)} samples to {fname}")

if __name__ == "__main__":
    data = scrape_slidescarnival_selenium(num_pages=3, delay=2)
    print("Total collected:", len(data))
    save_to_csv(data)


Scraping page 1 … URL: https://www.slidescarnival.com/category/free-templates/powerpoint-templates?page=1
Found 9 cards on page 1
Scraping page 2 … URL: https://www.slidescarnival.com/category/free-templates/powerpoint-templates?page=2
Found 9 cards on page 2
Scraping page 3 … URL: https://www.slidescarnival.com/category/free-templates/powerpoint-templates?page=3
Found 9 cards on page 3
Total collected: 27
✅ Saved 27 samples to slidescarnival_dataset.csv


# 라벨링

In [1]:
import gradio as gr
import os
import json
import random

# ===============================
# 기본 경로 설정
# ===============================
DATA_DIR = r"C:\Users\naeun\capstone\data"
IMAGE_DIR = os.path.join(DATA_DIR, "images")
LABEL_DIR = os.path.join(DATA_DIR, "labels")
os.makedirs(LABEL_DIR, exist_ok=True)

# 이미지 목록 불러오기 (jpg/png 지원)
image_files = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".png"))
])

# ===============================
# 샘플링: human labeling용 초기 샘플
# ===============================
NUM_INITIAL_LABEL = 300
initial_sample = random.sample(image_files, min(NUM_INITIAL_LABEL, len(image_files)))

# ===============================
# JSON 저장 + 다음 이미지 선택
# ===============================
def save_label_and_next(selected_image, layout, text_amount, color_contrast, visuals, consistency, feedback_text):
    if not selected_image:
        return gr.update(value=None), "❌ 이미지가 선택되지 않았습니다."

    json_path = os.path.join(LABEL_DIR, selected_image.rsplit(".",1)[0] + ".json")
    ppt_name = selected_image.rsplit(".",1)[0] + ".pptx"

    # 객체 피드백 파싱
    try:
        object_feedback = json.loads(feedback_text) if feedback_text.strip() else []
    except json.JSONDecodeError:
        return gr.update(value=selected_image), "❌ 객체 피드백 JSON 형식 오류!"

    data = {
        "filename": ppt_name,
        "overall_scores": {
            "layout": layout,
            "text_amount": text_amount,
            "color_contrast": color_contrast,
            "visuals": visuals,
            "consistency": consistency,
        },
        "object_feedback": object_feedback
    }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    # 자동 next
    next_img = get_next_image(selected_image)
    return gr.update(value=next_img), f"✅ '{selected_image}' 라벨 저장 완료!"

# ===============================
# 다음 이미지 선택
# ===============================
def get_next_image(current):
    idx = initial_sample.index(current) if current in initial_sample else 0
    if idx + 1 < len(initial_sample):
        return initial_sample[idx + 1]
    return initial_sample[0]  # 마지막이면 처음으로

# ===============================
# 이미지 로딩 (bytes)
# ===============================
from PIL import Image

def load_image(selected_image):
    if not selected_image:
        return None
    path = os.path.join(IMAGE_DIR, selected_image)
    # 1️⃣ 경로 그대로 반환
    return path
    # 2️⃣ 또는 PIL.Image로 열어서 반환 가능
    # return Image.open(path)

# ===============================
# Gradio UI
# ===============================
with gr.Blocks() as demo:
    gr.Markdown("# 🧩 Semi-Automatic 슬라이드 라벨링 툴")

    with gr.Row():
        selected_image = gr.Dropdown(
            choices=initial_sample,
            label="🎞 이미지 선택",
            value=initial_sample[0] if initial_sample else None
        )
        image_display = gr.Image(label="슬라이드 미리보기")

    with gr.Row():
        layout = gr.Slider(0, 1, value=0.5, step=0.1, label="Layout")
        text_amount = gr.Slider(0, 1, value=0.5, step=0.1, label="Text Amount")
        color_contrast = gr.Slider(0, 1, value=0.5, step=0.1, label="Color Contrast")
        visuals = gr.Slider(0, 1, value=0.5, step=0.1, label="Visuals")
        consistency = gr.Slider(0, 1, value=0.5, step=0.1, label="Consistency")

    gr.Markdown("### 🧾 객체별 피드백(JSON 배열)")

    feedback_text = gr.Textbox(
        lines=6,
        placeholder='[{"object_id":"shape_12","type":"rect","position":{"x":100,"y":200,"w":300,"h":100},"issue":"contrast low","suggestion":"increase contrast"}]'
    )

    save_btn = gr.Button("💾 저장 후 다음 이미지로 →")
    output_msg = gr.Markdown()

    # 이미지 선택 시 미리보기 업데이트
    selected_image.change(load_image, inputs=selected_image, outputs=image_display)

    # 저장 + 자동 next
    save_btn.click(
        save_label_and_next,
        inputs=[selected_image, layout, text_amount, color_contrast, visuals, consistency, feedback_text],
        outputs=[selected_image, output_msg]
    )

# ===============================
# 실행
# ===============================
if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=None)


c:\Users\naeun\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


### CLIP + Linear Probe
-> 자동 라벨링 학습

In [5]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import json
from pathlib import Path
import clip

# ===============================
# 경로 설정
# ===============================
DATA_DIR = Path(r"C:\Users\naeun\capstone\data")
IMAGE_DIR = DATA_DIR / "images"
LABEL_DIR = DATA_DIR / "labels"

# ===============================
# Dataset 정의
# ===============================
class SlideDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform

        # 라벨 JSON 있는 이미지만 사용
        self.images = []
        for f in image_dir.iterdir():
            if f.suffix.lower() in [".jpg", ".png"]:
                if (label_dir / f"{f.stem}.json").exists():
                    self.images.append(f)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        # JSON 라벨 읽기
        json_path = self.label_dir / f"{img_path.stem}.json"
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        scores = data["overall_scores"]
        label = torch.tensor([
            scores["layout"],
            scores["text_amount"],
            scores["color_contrast"],
            scores["visuals"],
            scores["consistency"]
        ], dtype=torch.float32)

        return image, label

# ===============================
# Transform, Dataset, DataLoader
# ===============================
device = "cuda" if torch.cuda.is_available() else "cpu"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073),
                         std=(0.26862954, 0.26130258, 0.27577711)),
])

dataset = SlideDataset(IMAGE_DIR, LABEL_DIR, transform=transform)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# ===============================
# CLIP + Linear Probe 모델 정의
# ===============================
clip_model, preprocess = clip.load("ViT-B/32", device=device)

# Linear Probe: CLIP의 이미지 임베딩 -> 5차원 점수
class LinearProbe(nn.Module):
    def __init__(self, clip_model, output_dim=5):
        super().__init__()
        self.clip_model = clip_model
        self.linear = nn.Linear(clip_model.visual.output_dim, output_dim)

    def forward(self, x):
        with torch.no_grad():  # CLIP freeze
            img_features = self.clip_model.encode_image(x)
        out = self.linear(img_features)
        return out

model = LinearProbe(clip_model).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.linear.parameters(), lr=1e-3)  # linear만 학습

# ===============================
# 학습 루프
# ===============================
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for imgs, labels in dataloader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader.dataset):.4f}")

# ===============================
# 모델 저장
# ===============================
torch.save(model.state_dict(), "clip_linear_probe.pth")
print("✅ Linear Probe 학습 완료, 저장됨!")


100%|███████████████████████████████████████| 338M/338M [00:36<00:00, 9.63MiB/s]


Epoch 1/10, Loss: 0.3580
Epoch 2/10, Loss: 0.0930
Epoch 3/10, Loss: 0.0783
Epoch 4/10, Loss: 0.0569
Epoch 5/10, Loss: 0.0483
Epoch 6/10, Loss: 0.0411
Epoch 7/10, Loss: 0.0363
Epoch 8/10, Loss: 0.0322
Epoch 9/10, Loss: 0.0289
Epoch 10/10, Loss: 0.0265
✅ Linear Probe 학습 완료, 저장됨!


### Auto Labeling

In [6]:
model = LinearProbe(clip_model).to(device)
model.load_state_dict(torch.load("clip_linear_probe.pth", map_location=device))
model.eval()

unlabeled_images = [f for f in IMAGE_DIR.iterdir() if f.suffix.lower() in [".jpg", ".png"]
                    and not (LABEL_DIR / f"{f.stem}.json").exists()]

with torch.no_grad():
    for img_path in unlabeled_images:
        image = preprocess(Image.open(img_path)).unsqueeze(0).to(device)
        scores = model(image).squeeze(0).cpu().tolist()

        data = {
            "filename": img_path.stem + ".pptx",
            "overall_scores": {
                "layout": float(scores[0]),
                "text_amount": float(scores[1]),
                "color_contrast": float(scores[2]),
                "visuals": float(scores[3]),
                "consistency": float(scores[4])
            },
            "object_feedback": []
        }

        json_path = LABEL_DIR / f"{img_path.stem}.json"
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

print("✅ 자동 라벨링 완료!")


✅ 자동 라벨링 완료!


In [ ]:
# test_clip_linear_probe.py
import torch
from PIL import Image
import clip

class CLIPLinearProbe(torch.nn.Module):
    def __init__(self, output_dim=5, device="cpu"):
        super().__init__()
        self.device = device
        self.clip_model, _ = clip.load("ViT-B/32", device=device, jit=False)
        self.clip_model.eval()
        self.fc = torch.nn.Linear(self.clip_model.visual.output_dim, output_dim)
        self.to(device)

    def forward(self, img):
        import torchvision.transforms as T
        transform = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.48145466, 0.4578275, 0.40821073],
                std=[0.26862954, 0.26130258, 0.27577711]
            )
        ])
        x = transform(img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            features = self.clip_model.encode_image(x)
            features = features / features.norm(dim=-1, keepdim=True)
            logits = self.fc(features)
        return features, logits

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = CLIPLinearProbe(device=device)

    # 학습된 Linear probe weight 로드
    probe_path = r"C:\Users\naeun\capstone\clip\clip_linear_probe.pth"
    state_dict = torch.load(probe_path, map_location=device)
    model.fc.load_state_dict(state_dict, strict=True)
    print(f"[INFO] Linear probe loaded from {probe_path}")

    # 테스트 이미지
    img_path = r"C:\Users\naeun\capstone\data\images\0_Blue-Green-and-Red-Professional-Consulting-Pitch-Deck-1.jpg"
    img = Image.open(img_path).convert("RGB")

    features, logits = model(img)
    print("[INFO] CLIP feature vector shape:", features.shape)
    print("[INFO] CLIP feature vector (first 10 dims):", features[0, :10].detach().cpu().numpy())
    print("[INFO] Linear probe logits:", logits[0].detach().cpu().numpy())

if __name__ == "__main__":
    main()


[INFO] Linear probe loaded from C:\Users\naeun\capstone\clip\clip_linear_probe.pth
[INFO] CLIP feature vector shape: torch.Size([1, 512])
[INFO] CLIP feature vector (first 10 dims): [-0.0212041   0.02089606  0.0204879  -0.01148572  0.01624417 -0.06425832
  0.04686058  0.0166695   0.03779343 -0.00503885]
[INFO] Linear probe logits: [-0.00111708  0.00931043  0.05042052  0.0654223  -0.00335912]
